# 📅 2026-04-29 (수) 개발 노트 : DB 데이터 정합성 복구 + FastAPI 49차원 추천 엔진 구축

## 🎯 오늘의 목표
- [x] 4,190개 게임 JSONL 데이터에 누락된 메타데이터(이름/장르/개발사) 복구
- [x] gem_potential 등 평가 지표 DB 적재
- [x] FastAPI 49차원 벡터 기반 추천 엔진 구축 (legacy 폐기, 신규 구조)
- [x] 60개 지표 기반 추천 알고리즘 검증 (by-game / by-preference)
- [x] 추천 점수 변별력 강화 (max_distance 튜닝)

## 🛠 진행 상황 및 핵심 기록

### 1️⃣ DB 데이터 정합성 복구
- `data/final/final_master_games.jsonl`에 **name/genres/developer가 전부 빈칸**으로 들어가는 문제 발견
- 원인: AI 배치 병합 시 원본 스팀 메타데이터(CSV)를 빼먹음
- `scripts/fix_data.py` 작성하여 `hidden_gem_data.csv`와 JSONL을 app_id 기준으로 병합
- 결과 파일: `data/final/final_master_games_fixed.jsonl`
- **추가 발견**: CSV에 `gem_potential`이 있었음! → `metrics.gem_potential`로 주입

### 2️⃣ Django DB 업데이트
- `python manage.py load_games --input ../data/final/final_master_games_fixed.jsonl --update`
- 4,190개 게임 전부 업데이트 성공
- 결과:
  - games 테이블: 이름/장르/개발사 모두 채워짐
  - game_metrics: gem_potential 4190/4190, confidence_score 4189/4190
  - 60개 지표 99.6% 완벽 데이터

### 3️⃣ FastAPI 신규 구조 구축 (legacy 폐기)
기존 33차원 코드를 `legacy/`로 이동하고 49차원 기준 신규 구조 작성:

```
fastapi_app/
├── main.py                  # FastAPI 진입점
├── config.py                # DB 설정 (juntae/0312)
├── database.py              # SQLAlchemy 비동기 엔진
├── models/game.py           # ORM + NUMERIC_METRIC_FIELDS(49)
├── schemas/game.py          # Pydantic v2 (ConfigDict)
├── services/recommender.py  # 49차원 추천 엔진 ⭐
└── routers/games.py         # API 엔드포인트
```

### 4️⃣ 추천 알고리즘 핵심 로직
- **벡터 변환**: GameMetric → 49차원 numpy array (None은 5.0 중립값)
- **하이브리드 유사도**: 코사인(50%) + 유클리드(50%)
- **max_distance = 30.0** (변별력 강화의 핵심)
- **가중치**: 유저 지정 지표는 2.5배 가중치
- **Hidden Gem 보너스**: gem_potential + 리뷰 적은 게임에 최대 0.15 보너스

### 5️⃣ 검증된 API 엔드포인트
- `GET /api/v1/games/stats/overview` → 4,190개 게임 통계 OK
- `GET /api/v1/games/{app_id}` → Counter-Strike 2 (app_id=730) 60지표 정상
- `GET /api/v1/games/search?q=...` → 검색 정상
- `POST /api/v1/games/recommend/by-game` → CS2 → CS/Insurgency/CS:Source 매칭 (0.99~0.95)
- `POST /api/v1/games/recommend/by-preference` → 전략 게임 검색 → Divinity OS 2/Armello/Frosthaven 매칭 (0.79~0.76)

### 📌 기억해야 할 정보
- **DB 접속**: `docker exec -it hidden_gem_db psql -U juntae -d hidden_gem_db`
- **FastAPI 서버 실행**: `python main.py` (8000 포트)
- **API 문서**: http://localhost:8000/docs
- **신규 데이터 파일**: `data/final/final_master_games_fixed.jsonl`
- **49차원 지표 리스트**: `fastapi_app/models/game.py` → `NUMERIC_METRIC_FIELDS`
- **9개 태그 리스트**: `BOOLEAN_TAG_FIELDS`

## 🚨 트러블슈팅 (문제 및 해결)

### 문제 1: games 테이블에 이름/장르가 전부 빈칸
- **문제:** `SELECT name, genres FROM games LIMIT 5;` 결과가 전부 빈 문자열
- **원인:** `merge_metrics.py`가 AI 분석 결과만 JSONL로 저장하고 원본 스팀 메타데이터(CSV)를 병합하지 않음. 즉 JSONL 최상위에 name/genres 키 자체가 없었음.
- **해결:** `scripts/fix_data.py` 작성
  - `pd.read_csv('hidden_gem_data.csv')` → `set_index('app_id').to_dict('index')`로 빠른 검색용 딕셔너리 생성
  - JSONL 순회하며 app_id 기준 매칭, name/genres/developer/description/header_image 등 주입
  - `final_master_games_fixed.jsonl`로 출력
  - `python manage.py load_games --update` 로 4,190개 일괄 업데이트

### 문제 2: gem_potential이 metrics에 없음
- **문제:** AI 분석 JSONL에 `gem_potential` 필드가 누락됨 (토큰 다이어트 영향)
- **원인:** GPT-5.4가 신규 18개 지표만 추출하면서 평가 지표를 응답에서 뺌
- **해결:** CSV에 `gem_potential` 컬럼이 이미 있어서 `fix_data.py`에서 `data['metrics']['gem_potential']`로 주입. 결과적으로 4,190개 전부 채워짐 ✅

### 문제 3: FastAPI 추천 점수가 전부 1.0으로 동일
- **문제:** 상위 5개 추천 게임의 `similarity_score`가 모두 1.0 → 변별력 없음
- **원인:** `_euclidean_similarity()`의 `max_distance = 70` (sqrt(49*100))이 너무 커서 실제 게임 간 거리(보통 10~30)가 0.7~1.0 사이에 압축됨
- **해결:**
  - `max_distance = 30.0`으로 축소
  - 코사인/유클리드 비율 6:4 → 5:5로 조정
  - 가중치 2.0 → 2.5로 강화
  - Hidden Gem 보너스 0.3 → 0.15로 축소 (점수 영향 줄임)
  - 결과: 0.7925 / 0.7694 / 0.7689 / 0.7642 / 0.7641 → **변별력 확보** ✅

### 문제 4: by-game 추천에서 match_reasons가 비어있음
- **문제:** CS2 추천 결과의 `match_reasons: []`
- **원인:** 기존 `_generate_match_reasons()`는 `preferences` 인자만 받음 → by-game에는 preferences가 없으니 빈 리스트 반환
- **해결:**
  - `_generate_match_reasons_from_preferences()`: 선호도 기반 (기존)
  - `_generate_match_reasons_from_target()`: **기준 게임과 후보 게임의 공통 극단값** 분석 (신규)
  - 두 게임 모두 7+ 이거나 둘 다 ≤3인 지표를 '공통적으로 높음/낮음'으로 표시
  - 결과: CS2 추천에서 'reflex_demand 공통적으로 높음 (9.0)' 같은 인사이트 나옴 ✅

### 문제 5: psql 명령어를 bash에 직접 입력해서 syntax error
- **문제:** `\dt`, `SELECT COUNT(*) FROM games;`를 bash에 입력 → `bash: SELECT: command not found`
- **원인:** psql 접속 안 한 상태에서 SQL 입력
- **해결:** `docker exec -i hidden_gem_db psql -U juntae -d hidden_gem_db << 'EOF' ... EOF` 형식으로 heredoc 사용

## 💡 인사이트 및 다음 할 일

### 배운 점
1. **데이터 파이프라인 검증의 중요성**: AI 분석 결과만 보관하고 원본 메타데이터를 빼먹으면 나중에 큰 작업 필요. 처음부터 병합 설계 필수.
2. **벡터 유사도의 정규화 튜닝**: 이론상 최대 거리(70)와 실제 데이터 분포(10~30)는 다름. 실측 기반 튜닝 필요.
3. **추천 이유 생성의 두 가지 패턴**:
   - 선호도 기반: '유저가 원한 값과 얼마나 가까운가'
   - 게임 기반: '두 게임이 어떤 점에서 공통적인가'
4. **FastAPI + SQLAlchemy 2.0 비동기 패턴**: `selectinload`로 N+1 쿼리 방지, `async_sessionmaker`로 세션 관리
5. **Pydantic v2 마이그레이션**: `class Config` → `ConfigDict(from_attributes=True)` 통일 필요
6. **legacy 폴더 활용**: 구버전 코드 폐기 대신 보관 → 롤백/참고 가능

### 다음 할 일 (우선순위)

#### 🥇 1순위: 헤더 이미지 채우기 (1시간 이내)
- 현재 `header_image` 필드 전부 비어있음
- Steam CDN URL 패턴 활용: `https://cdn.cloudflare.steamstatic.com/steam/apps/{app_id}/header.jpg`
- `scripts/fill_header_images.py` 작성하여 DB 일괄 업데이트
- 프론트엔드 개발 준비 완료 상태로 만들기

#### 🥈 2순위: 프론트엔드 시작 (Next.js)
- API 다 됐으니 UI만 만들면 됨
- 추천 결과 시각화: 육각형 스탯, 매칭 이유 카드, 게임 헤더 이미지
- 검색 + 필터 UI (cozy_factor 슬라이더 등)

#### 🥉 3순위: 유저 시스템 구축
- 로그인 (소셜 OAuth)
- 좋아요/싫어요 → taste_dna 학습
- 추천 히스토리 저장

#### 🏅 4순위: 추천 알고리즘 고도화
- Two-Tower 임베딩 학습
- 다양성(diversity) 보장 (MMR 알고리즘)
- 협업 필터링 추가

## 📊 오늘의 통계

| 항목 | 수치 |
|------|------|
| 처리한 게임 수 | 4,190개 |
| 복구한 메타데이터 필드 | 11개 (name, genres, developer, ...) |
| 적재 완료 지표 | 60개 (49 수치 + 9 태그 + 2 평가) |
| 작성한 신규 파일 | 12개 (FastAPI 구조 전체) |
| 폐기한 legacy 파일 | 33차원 기반 구버전 일체 |
| API 엔드포인트 검증 | 6개 모두 정상 |
| 추천 점수 변별력 | 1.0 (전부 동일) → 0.99~0.76 (개선) |

## 🔗 핵심 명령어 치트시트

```bash
# DB 접속
docker exec -it hidden_gem_db psql -U juntae -d hidden_gem_db

# 데이터 정합성 복구
cd /c/Hidden-Gem-project/scripts
python fix_data.py

# Django 데이터 업데이트
cd /c/Hidden-Gem-project/django_core
source ../.venv/Scripts/activate
python manage.py load_games --input ../data/final/final_master_games_fixed.jsonl --update

# FastAPI 서버 실행
cd /c/Hidden-Gem-project/fastapi_app
source ../.venv/Scripts/activate
python main.py

# API 테스트 (Swagger)
# → http://localhost:8000/docs
```